# NC-Musical: AI Music Transcription (Direct Colab Pipeline & Web Editor)
Run GPU-accelerated automatic music transcription (AMT) using the MuScriptor engine directly inside Google Colab.

**Features:**
- Transcribe any **YouTube URL** or **Uploaded Audio/Video File** directly in Colab.
- Extract multi-track MIDI notes (Piano, Guitar, Bass, Drums) powered by GPU.
- Synthesize and play audio previews directly in the notebook using `MS Basic.sf3` SoundFont.
- Download `.mid` MIDI files directly to your computer.
- Optionally launch the interactive **Piano Roll Web Editor GUI**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentHitmanFaris/NC-Musical/blob/Stable/NC_Musical_Colab.ipynb)

In [ ]:
# @title 1. Mount Google Drive (Persistent Model Storage)
import os
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

# Directory in Google Drive for caching models permanently
drive_models_dir = "/content/drive/MyDrive/NC-Musical-Models"
os.makedirs(drive_models_dir, exist_ok=True)
hf_cache_dir = os.path.join(drive_models_dir, "huggingface")
os.makedirs(hf_cache_dir, exist_ok=True)

# Set HuggingFace and PyTorch cache dirs to Google Drive
os.environ["HF_HOME"] = hf_cache_dir
os.environ["TORCH_HOME"] = os.path.join(drive_models_dir, "torch")

print(f"All AI models will be saved permanently to: {drive_models_dir}")


In [ ]:
# @title 2. Check GPU, Tokens & Setup Repository
import os
import sys
import subprocess
import getpass

print("Checking GPU environment...")
!nvidia-smi

# Configure HuggingFace Token (HF_TOKEN)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    print("\nNotice: MuScriptor models on HuggingFace require a HuggingFace User Token (HF_TOKEN).")
    print("If saved in Colab Secrets as 'HF_TOKEN', it is loaded automatically. Otherwise, enter below:")
    try:
        hf_token = getpass.getpass("HuggingFace Token (HF_TOKEN): ").strip()
    except Exception:
        hf_token = ""

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HuggingFace Access Token configured successfully!")

print("\nInstalling system dependencies (FFmpeg and FluidSynth)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fluidsynth

print("\nCloning / Updating NC-Musical repository...")
repo_dir = "/content/NC-Musical"
if not os.path.exists(repo_dir):
    res = subprocess.run(["git", "clone", "https://github.com/AgentHitmanFaris/NC-Musical.git", repo_dir], capture_output=True, text=True)
    if res.returncode != 0:
        print("Public clone failed (Repository is private). Access Token required.")
        token = None
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
        
        if not token:
            print("\nPlease enter your GitHub Personal Access Token (PAT) to clone the private repository:")
            token = getpass.getpass("GitHub Token: ")
        
        token = token.strip()
        !git clone https://{token}@github.com/AgentHitmanFaris/NC-Musical.git /content/NC-Musical

if os.path.exists(repo_dir):
    %cd /content/NC-Musical
    !git pull || true
    print("\nInstalling Python dependencies...")
    !pip install -U -q yt-dlp uvicorn fastapi soundfile torch torchvision torchaudio
    !pip install -q muscriptor || true

    print("\nRunning patch script...")
    !python patch_muscriptor.py || true
else:
    print("Error: Could not clone repository.")


In [ ]:
# @title 3. Setup MS Basic.sf3 SoundFont in Google Drive
import os
import urllib.request

drive_models_dir = "/content/drive/MyDrive/NC-Musical-Models"
drive_sf3_path = os.path.join(drive_models_dir, "MS Basic.sf3")
local_sf3_path = "/content/NC-Musical/MS Basic.sf3"

if not os.path.exists(drive_sf3_path):
    print("Downloading MS Basic.sf3 SoundFont (~50MB) to Google Drive...")
    sf3_urls = [
        "https://raw.githubusercontent.com/musescore/MuseScore/v4.1.0/share/sound/MS%20Basic.sf3",
        "https://huggingface.co/MuScriptor/assets/resolve/main/MuseScore_General.sf3"
    ]
    downloaded = False
    for url in sf3_urls:
        try:
            urllib.request.urlretrieve(url, drive_sf3_path)
            print("MS Basic.sf3 downloaded and saved to Google Drive!")
            downloaded = True
            break
        except Exception as e:
            print(f"Notice ({e}): trying next source...")
    if not downloaded:
        print("Warning: SoundFont download failed.")
else:
    print("MS Basic.sf3 SoundFont found in Google Drive!")

if os.path.exists(drive_sf3_path) and not os.path.exists(local_sf3_path):
    !cp "$drive_sf3_path" "$local_sf3_path"


In [ ]:
# @title 4. Direct AI Music Transcription (Colab Interactive Pipeline)
# @markdown Choose your audio input source and model options below, then click **Run**.
import os
import sys
import time
import torch
from pathlib import Path
from IPython.display import Audio, display
import google.colab.files

# Force upgrade yt_dlp to bypass latest YouTube bot protection
!pip install -U -q yt-dlp music21
import yt_dlp

input_source = "YouTube URL" # @param ["YouTube URL", "Upload Audio/Video File"]
youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ" # @param {type:"string"}
model_size = "large" # @param ["small", "medium", "large"]
target_instruments = "" # @param {type:"string"} Leave empty for ALL instruments
export_chord_names = True # @param {type:"boolean"}

audio_path = "/content/input_audio.wav"
if os.path.exists(audio_path):
    os.remove(audio_path)

def download_youtube(url, out_tmpl):
    """Try multiple cookieless player clients, then fall back to cookies if present."""

    # Clients ordered by likelihood of bypassing bot-check without auth
    cookieless_strategies = [
        ['tv_embedded'],
        ['android'],
        ['android_vr'],
        ['ios'],
        ['tv_embedded', 'android'],
        ['android', 'ios'],
        ['mweb'],
        ['web_embedded'],
        ['web', 'android'],
    ]

    cookie_paths = [
        "/content/cookies.txt",
        "/content/drive/MyDrive/cookies.txt",
        "/content/drive/MyDrive/NC-Musical-Models/cookies.txt",
    ]
    found_cookie = next((cp for cp in cookie_paths if os.path.exists(cp)), None)

    base_opts = {
        'format': 'bestaudio/best',
        'outtmpl': out_tmpl,
        'quiet': True,
        'no_warnings': True,
        'noplaylist': True,
    }

    last_err = None

    # ── Phase 1: cookieless attempts ──────────────────────────────────────────
    for clients in cookieless_strategies:
        opts = {**base_opts, 'extractor_args': {'youtube': {'player_client': clients}}}
        try:
            print(f"  Trying client={clients} (no cookies)...", end=" ")
            with yt_dlp.YoutubeDL(opts) as ydl:
                ydl.download([url])
            files = list(Path(out_tmpl).parent.glob("yt_temp.*"))
            if files:
                print("✓")
                return True, None
            print("✗ (no file)")
        except Exception as e:
            last_err = e
            print(f"✗")

    # ── Phase 2: with cookies (if file exists) ────────────────────────────────
    if found_cookie:
        print(f"\n  All cookieless attempts failed. Trying with cookies: {found_cookie}")
        for clients in [['tv_embedded'], ['android'], ['web']]:
            opts = {**base_opts,
                    'extractor_args': {'youtube': {'player_client': clients}},
                    'cookiefile': found_cookie}
            try:
                print(f"  Trying client={clients} + cookies...", end=" ")
                with yt_dlp.YoutubeDL(opts) as ydl:
                    ydl.download([url])
                files = list(Path(out_tmpl).parent.glob("yt_temp.*"))
                if files:
                    print("✓")
                    return True, None
                print("✗")
            except Exception as e:
                last_err = e
                print("✗")

    return False, last_err


if input_source == "YouTube URL":
    print(f"Fetching YouTube audio: {youtube_url}")
    print("-" * 50)
    for f in Path("/content").glob("yt_temp.*"):
        try: os.remove(f)
        except: pass

    success, err = download_youtube(youtube_url, '/content/yt_temp.%(ext)s')

    dl_files = list(Path("/content").glob("yt_temp.*"))
    if dl_files:
        raw_audio = str(dl_files[0])
        print(f"\nConverting {Path(raw_audio).name} to 16kHz mono WAV...")
        !ffmpeg -y -i "$raw_audio" -ac 1 -ar 16000 "$audio_path" -loglevel quiet
        try: os.remove(raw_audio)
        except: pass
        print("Audio ready!")
    else:
        print("\n" + "!" * 70)
        print("ERROR: All download strategies failed.")
        print(f"Last error: {err}")
        print("\nFix options:")
        print("  1. Upload a cookies.txt file to /content/ (from your browser)")
        print("  2. Change input_source to 'Upload Audio/Video File'")
        print("!" * 70 + "\n")

else:
    print("Please click 'Choose Files' below to upload an audio or video file (MP3, WAV, MP4, M4A, FLAC):")
    uploaded = google.colab.files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        print(f"Converting {fname} to 16kHz mono WAV...")
        !ffmpeg -y -i "$fname" -ac 1 -ar 16000 "$audio_path" -loglevel quiet

if not os.path.exists(audio_path):
    print("Error: Audio source could not be loaded.")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nLoading MuScriptor AI Model ('{model_size}' on {device})...")
    from muscriptor.transcription_model import TranscriptionModel
    from muscriptor.utils.audio import _read_wav_file
    from muscriptor.events import ProgressEvent
    from muscriptor.utils.auralization import synthesize

    model = TranscriptionModel.load_model(weights_path=model_size, device=device)

    with open(audio_path, "rb") as f:
        wav, sr = _read_wav_file(f)

    # Instrument resolution — empty = ALL instruments
    INST_ALIASES = {
        "piano": "acoustic_piano", "guitar": "acoustic_guitar",
        "electric_guitar": "clean_electric_guitar", "bass": "electric_bass",
        "drums": "drums", "drum": "drums", "percussion": "drums",
        "strings": "string_ensemble", "brass": "brass_section",
        "sax": "soprano_and_alto_sax", "saxophone": "soprano_and_alto_sax",
        "flute": "flutes", "synth": "synth_lead", "pad": "synth_pad",
        "vocal": "voice", "vocals": "voice",
    }
    raw_insts = [i.strip() for i in target_instruments.split(",") if i.strip()]
    if raw_insts:
        inst_list = [INST_ALIASES.get(k.lower().replace(" ", "_"), k) for k in raw_insts]
        print(f"Transcribing selected tracks: {inst_list}")
    else:
        inst_list = None
        print("Transcribing ALL instruments...")

    # ── Progress bar ──────────────────────────────────────────────────────────
    def _fmt_time(s):
        return f"{int(s//60)}m {int(s%60):02d}s" if s >= 60 else f"{s:.0f}s"

    def _draw_progress(completed, total, elapsed, note_count):
        pct = int(completed / total * 100) if total > 0 else 0
        bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
        eta = (elapsed / max(completed, 1)) * (total - completed) if completed > 0 else 0
        print(f"\r  [{bar}] {pct:3d}%  |  Elapsed: {_fmt_time(elapsed)}  |  ETA: {_fmt_time(eta)}  |  Notes: {note_count}   ",
              end="", flush=True)

    print()
    events = []
    start_t = time.time()
    devnull = open(os.devnull, 'w')
    real_stdout = sys.stdout
    try:
        for ev in model.transcribe((wav, sr), instruments=inst_list, batch_size=1):
            sys.stdout = real_stdout
            if isinstance(ev, ProgressEvent):
                if ev.total > 0:
                    _draw_progress(ev.completed, ev.total, time.time() - start_t, len(events))
            else:
                events.append(ev)
            sys.stdout = devnull
    finally:
        sys.stdout = real_stdout
        devnull.close()

    elapsed = time.time() - start_t
    print(f"\n\n✅ Transcription complete in {_fmt_time(elapsed)} — {len(events)} note events extracted.")

    # Save MIDI
    midi_bytes = model.events_to_midi_bytes(iter(events))
    midi_filename = "/content/transcription_result.mid"
    with open(midi_filename, "wb") as f:
        f.write(midi_bytes)
    print(f"MIDI saved: {midi_filename}")

    # ── Chord Analysis ────────────────────────────────────────────────────────
    chord_filename = "/content/chords.txt"
    if export_chord_names:
        print("\nAnalysing chords with music21...")
        try:
            from music21 import converter, chord as m21chord
            score = converter.parse(midi_filename)
            chordified = score.chordify()
            lines = ["CHORD ANALYSIS — NC-Musical Transcription", "=" * 50]
            prev_label = None
            for c in chordified.flatten().getElementsByClass(m21chord.Chord):
                offset_sec = float(c.offset)
                ts = f"{int(offset_sec//60)}:{offset_sec%60:05.2f}"
                try:
                    label = f"{c.root().name} {c.commonName}" if c.root() else " ".join(p.name for p in c.pitches)
                except Exception:
                    label = " ".join(p.name for p in c.pitches)
                if label != prev_label:
                    lines.append(f"[{ts}]  {label}")
                    prev_label = label
            chord_text = "\n".join(lines)
            with open(chord_filename, "w", encoding="utf-8") as f:
                f.write(chord_text)
            print("\n" + chord_text[:2000])
            if len(chord_text) > 2000:
                print(f"... (see {chord_filename})")
        except Exception as e:
            print(f"Chord analysis notice: {e}")

    # ── Audio Preview ─────────────────────────────────────────────────────────
    preview_wav = "/content/transcription_preview.wav"
    sf3_path = "/content/NC-Musical/MS Basic.sf3"
    try:
        print("\nSynthesizing audio preview...")
        synthesize(midi_filename, preview_wav, soundfont_path=sf3_path if os.path.exists(sf3_path) else None)
        display(Audio(preview_wav))
    except Exception as e:
        print("Audio preview notice:", e)

    # ── Download ──────────────────────────────────────────────────────────────
    print("\nDownloading files...")
    google.colab.files.download(midi_filename)
    if export_chord_names and os.path.exists(chord_filename):
        google.colab.files.download(chord_filename)


In [ ]:
# @title 5. (Optional) Launch Piano Roll Web Editor GUI
import os
import time
import subprocess
import torch

# Kill any existing server or tunnel instances
!pkill -f "server_gui.py" || true
!pkill -f "cloudflared" || true

PORT = 8222
device_arg = "cuda" if torch.cuda.is_available() else "cpu"

print("Starting MuScriptor FastAPI server on GPU...")
env = os.environ.copy()

server_process = subprocess.Popen(
    ["python", "server_gui.py", "--port", str(PORT), "--model", "large", "--device", device_arg],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

time.sleep(3)

# Launch Cloudflare Tunnel to expose port 8222 directly over HTTPS
print("\nCreating direct HTTPS Public Link via Cloudflare Tunnel...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

cf_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
time.sleep(3)
for _ in range(30):
    line = cf_process.stdout.readline()
    if "trycloudflare.com" in line:
        parts = line.strip().split()
        for part in parts:
            if "trycloudflare.com" in part:
                public_url = part
                if not public_url.startswith("http"):
                    public_url = "https://" + public_url
                break
        if public_url:
            break
    time.sleep(0.5)

if public_url:
    public_url = public_url.rstrip("/") + "/"
    print("\n" + "="*65)
    print("SUCCESS! Click the link below to open the MuScriptor Web GUI:")
    print(f"--> {public_url}index.html <--")
    print("="*65 + "\n")
else:
    print("Cloudflare Tunnel starting... Check logs below for link.")

print("Streaming backend logs (Keep this cell running while using the Web GUI):")
print("-" * 65)

try:
    for line in iter(server_process.stdout.readline, ''):
        if line:
            print(line, end='')
except KeyboardInterrupt:
    print("\nStopping server...")
    server_process.terminate()
    cf_process.terminate()
